In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
user_bronze = spark.table("nutrifit_catalog.bronze.users_profiles")
display(user_bronze.limit(5))

In [0]:
user_no_name = user_bronze.filter(user_bronze["name"].isNull())
print(f"total rows: {user_bronze.count()} and null names: {user_no_name.count()}")


In [0]:
##users without no name and user_id cast to int
user_silver = user_bronze.dropna(subset=["name"]).withColumn("user_id", F.col("user_id").cast("int"))

In [0]:
user_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nutrifit_catalog.silver.users")
print(f"Bronze rows: {user_bronze.count()} and silver rows: {user_silver.count()}")

In [0]:
%sql
select count(*) as total_rows, count(name) as rows_with_name
from nutrifit_catalog.silver.users

In [0]:
spark.table("nutrifit_catalog.silver.users").printSchema()

In [0]:
activity_bronze = spark.table("nutrifit_catalog.bronze.daily_activity")
users_silver = spark.table("nutrifit_catalog.silver.users")

#remove duplicates, set calories to be positive, rename columns "id" and "ActivityDate", join with users 
activity_silver = (
    activity_bronze
    .dropDuplicates()
    .filter(F.col("Calories") >= 0)
    .withColumnRenamed("id", "user_id").withColumnRenamed("ActivityDate", "activity_date")
    .join(users_silver, on="user_id", how="left_semi")
)

activity_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nutrifit_catalog.silver.daily_activity")

print(f"Bronze rows {activity_bronze.count()} and silver rows {activity_silver.count()}")

In [0]:
nutrition_bronze = spark.table("nutrifit_catalog.bronze.daily_nutrition")

#remove duplicates
nutrition_dedup= nutrition_bronze.dropDuplicates()

#check orphans rows
orphans = nutrition_dedup.join(users_silver, on="user_id", how="left_anti")
print(f"orphan rows {orphans.count()}")

nutrition_silver = nutrition_dedup.join(users_silver, on="user_id", how="left_semi")

In [0]:
nutrition_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("nutrifit_catalog.silver.daily_nutrition")
print(f"Bronze rows: {nutrition_bronze.count()} and silver rows: {nutrition_silver.count()}")